# 🌊 OceanWatch: YOLO vs RT-DETR for underwater marine litter detection
**MAAI901 Deep Learning for Visual Data · University of Wollongong in Dubai**

This notebook compares a CNN detector (**YOLO26-l**) with a transformer detector (**RT-DETR-l**) for finding litter in underwater robot (ROV) images from the **TrashCan 1.0** dataset (Hong, Fulton & Sattar, 2020).

| Step | What it does | GPU? | Time |
|---|---|---|---|
| 1 | Setup: GPU check, Google Drive, find files, unzip, install | – | ~3 min |
| 2 | Build the 3-class dataset (trash / bio / rov) with a **video-level** split | – | ~1 min |
| 3 | Train YOLO26-l (skipped if already trained) | ✔ | ~2.3 h |
| 4 | Train RT-DETR-l (skipped if already trained) | ✔ | ~3.8 h |
| 5 | Final test: accuracy + speed on 1,081 unseen images | ✔ | ~10 min |
| 6 | Experiments: clear vs challenging images, image enhancement | ✔ | ~15 min |
| 7 | Figures: where the two models differ | ✔ | ~5 min |
| 8 | **OceanWatch app**: upload an image → detections, confidence, accuracy | – | ~1 min |
| 9 | Export files for the Streamlit app / GitHub | – | ~1 min |

**How to run:** `Runtime → Change runtime type → T4 GPU → Save`, then `Runtime → Run all`. Every result is saved to Google Drive → `OceanWatch`, so finished steps are skipped next time. Without a GPU, select Step 8 and use `Runtime → Run before`, then run Step 8: the app works on CPU.

**Files needed in My Drive:** the TrashCan `dataset.zip` (any name, ~528 MB), `video_split.json`, and the trained models if you do not want to retrain.

## 1. Setup
Checks for a GPU, connects Google Drive, finds the dataset zip, the split file and the trained YOLO model wherever they are in My Drive, unzips the dataset and installs **Ultralytics 8.4.177** (the version both models were trained with).

In [ ]:
# ===== 1. GPU check, Google Drive, find your files, unzip, install =====
import os, sys, subprocess, torch
from pathlib import Path

if torch.cuda.is_available():
    print('GPU OK:', torch.cuda.get_device_name(0))
else:
    print('No GPU: fine for the app (Cell 5) and for showing saved results. Training and the experiments need a T4 GPU.')

from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
WORK = Path('/content')

def find(pattern, max_depth=4):
    """Search My Drive (up to 4 folders deep) for files matching a pattern."""
    hits = []
    for root, dirs, files in os.walk(DRIVE):
        if len(Path(root).relative_to(DRIVE).parts) >= max_depth:
            dirs[:] = []
        hits += [Path(root) / f for f in files if Path(f).match(pattern)]
    return sorted(hits)

zips = [p for p in find('*.zip') if 450e6 < p.stat().st_size < 600e6]
assert zips, f'Dataset zip (~528 MB) not found in My Drive. Zip files found: {[p.name for p in find("*.zip")]}'
ZIP = sorted(zips, key=lambda p: 'plastic' not in p.name.lower())[0]

splits = find('video_split*.json')
assert splits, 'video_split.json not found in My Drive - upload it from the OceanWatch folder of your first account.'
SPLIT_FILE = splits[0]

yolo_best = [p for p in find('best*.pt') if 'rtdetr' not in str(p)]
print('Dataset zip :', ZIP)
print('Split file  :', SPLIT_FILE)
print('YOLO model  :', yolo_best[0] if yolo_best else 'not found (only needed later, for the final test)')

if not list((WORK / 'trashcan').rglob('instance_version')):
    print('Unzipping the dataset (1-5 minutes)...')
    subprocess.run(['unzip', '-oq', str(ZIP), '-d', str(WORK / 'trashcan')], check=True)
SRC = next((WORK / 'trashcan').rglob('instance_version'))
print('Dataset folder:', SRC)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.177'], check=True)  # same version as the YOLO run
import ultralytics
print('Ultralytics', ultralytics.__version__)

## 2. Dataset: 3 classes, video-level split
TrashCan's images are frames taken once per second from ROV videos, so neighbouring frames are near-duplicates. The official split puts frames of **113 of its 115 test videos** into training as well, which inflates test scores. We therefore assign **whole videos** to train / val / test (70 / 15 / 15), balanced per class, and save the split in `video_split.json` so every run uses exactly the same images.

The 22 fine-grained TrashCan classes are merged into the annotators' three top-level labels: **trash**, **bio** (plants and animals) and **rov** (the robot's own equipment). A 22-class YOLO run reached mAP50 0.970 on training videos but only 0.139 on unseen ones (0.382 if the class is ignored), so the fine-grained labels did not generalise.

The cell checks that the counts match the dataset the models were trained on.

In [ ]:
# ===== 2. Build the 3-class dataset with the SAME video split as the YOLO run =====
import json, re, shutil, collections, yaml

OUT = WORK / 'trashcan_yolo3'
NAMES = ['trash', 'bio', 'rov']        # same class order as the YOLO 3-class run
EXPECTED = {'train': {'images': 5045, 'trash': 4176, 'bio': 1954, 'rov': 2307},
            'val':   {'images': 1086, 'trash': 954,  'bio': 353,  'rov': 405},
            'test':  {'images': 1081, 'trash': 876,  'bio': 498,  'rov': 605}}

def video_of(file_name):
    m = re.match(r'(vid_\d+)', os.path.basename(file_name))
    return m.group(1) if m else Path(file_name).stem

def coarse(name):                      # TrashCan's top-level labels
    return 2 if name == 'rov' else 0 if name.startswith('trash') else 1

assign = json.loads(SPLIT_FILE.read_text())
if OUT.exists():
    shutil.rmtree(OUT)
stats = {s: collections.Counter() for s in EXPECTED}

for folder in ('train', 'val'):        # pool the official train + val files
    coco = json.loads((SRC / f'instances_{folder}_trashcan.json').read_text())
    cls = {c['id']: coarse(c['name']) for c in coco['categories']}
    anns = collections.defaultdict(list)
    for a in coco['annotations']:
        anns[a['image_id']].append(a)
    for im in coco['images']:
        v = video_of(im['file_name'])
        assert v in assign, f'{v} is not in {SPLIT_FILE.name} - wrong split file?'
        split = assign[v]
        src = SRC / folder / os.path.basename(im['file_name'])
        img_dir, lbl_dir = OUT / 'images' / split, OUT / 'labels' / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        try:
            os.link(src, img_dir / src.name)
        except OSError:
            shutil.copy2(src, img_dir / src.name)
        W, H = im['width'], im['height']
        lines = []
        for a in anns[im['id']]:
            x, y, w, h = a['bbox']
            x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
            if x1 - x0 < 1 or y1 - y0 < 1:
                continue
            k = cls[a['category_id']]
            lines.append(f'{k} {(x0 + x1) / 2 / W:.6f} {(y0 + y1) / 2 / H:.6f} '
                         f'{(x1 - x0) / W:.6f} {(y1 - y0) / H:.6f}')
            stats[split][NAMES[k]] += 1
        (lbl_dir / f'{src.stem}.txt').write_text('\n'.join(lines))
        stats[split]['images'] += 1

(OUT / 'data.yaml').write_text(yaml.safe_dump(
    {'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'test': 'images/test',
     'names': dict(enumerate(NAMES))}, sort_keys=False))

print(f"{'split':<6}{'images':>8}{'trash':>7}{'bio':>6}{'rov':>6}")
for s in EXPECTED:
    print(f'{s:<6}' + ''.join(f'{stats[s][k]:>{w}}' for k, w in [('images', 8), ('trash', 7), ('bio', 6), ('rov', 6)]))
assert all(stats[s][k] == n for s in EXPECTED for k, n in EXPECTED[s].items()), \
    'Counts differ from the YOLO run - wrong video_split.json? Do not train; send Claude this output.'
print('\nIdentical to the dataset YOLO was trained on ✓')

## 3. YOLO26-l (CNN)
Single-stage CNN detector (Ultralytics, January 2026), COCO-pretrained. Same image size, batch, seed and data as RT-DETR; Ultralytics' automatic optimiser choice. Skipped when `best_yolo26l_3cls.pt` (or another YOLO `best.pt`) is in My Drive.

In [ ]:
# ===== 3. Train YOLO26-l (CNN) - skipped if the trained model is already in Drive =====
from ultralytics import YOLO

if yolo_best:
    print('YOLO26-l already trained - using', yolo_best[0])
else:
    assert torch.cuda.is_available(), 'Training needs a GPU: Runtime -> Change runtime type -> T4 GPU.'
    YOLO('yolo26l.pt').train(
        data=str(OUT / 'data.yaml'), epochs=40, imgsz=640, batch=16, seed=0, deterministic=False, workers=2,
        optimizer='auto', project=str(DRIVE / 'OceanWatch' / 'runs'), name='yolo26l-3cls', exist_ok=True)
    yolo_best = [DRIVE / 'OceanWatch' / 'runs' / 'yolo26l-3cls' / 'weights' / 'best.pt']

## 4. RT-DETR-l (transformer)
Real-Time DEtection TRansformer (Zhao et al., 2024), COCO-pretrained. Same image size, batch, seed and data as YOLO, with the AdamW recipe of the original paper (lr 1e-4, weight decay 1e-4, no bias warm-up). Saved to Drive after every epoch; if Colab disconnects, run again and it resumes.

In [ ]:
# ===== 4. Train RT-DETR-l (transformer) with the same settings as YOLO =====
from ultralytics import RTDETR

DATA = str(OUT / 'data.yaml')
RUNS = DRIVE / 'OceanWatch' / 'runs'   # saved to Drive after every epoch
NAME, EPOCHS = 'rtdetr-l-3cls', 40
run = RUNS / NAME
last, csv = run / 'weights' / 'last.pt', run / 'results.csv'
done = len(csv.read_text().strip().splitlines()) - 1 if csv.exists() else 0

if done >= EPOCHS:
    print(f'{NAME}: already finished ({done} epochs)')
elif last.exists():
    assert torch.cuda.is_available(), 'Training needs a GPU: Runtime -> Change runtime type -> T4 GPU.'
    print(f'{NAME}: resuming after epoch {done}')
    RTDETR(str(last)).train(resume=True)
else:
    assert torch.cuda.is_available(), 'Training needs a GPU: Runtime -> Change runtime type -> T4 GPU.'
    print(f'{NAME}: starting a new {EPOCHS}-epoch run')
    RTDETR('rtdetr-l.pt').train(
        data=DATA, epochs=EPOCHS, imgsz=640, batch=16, seed=0, deterministic=False, workers=2,  # identical to YOLO
        optimizer='AdamW', lr0=1e-4, weight_decay=1e-4, warmup_bias_lr=0.0,                 # RT-DETR recipe
        project=str(RUNS), name=NAME, exist_ok=True)

## 5. Final test
Both models' best checkpoints (chosen on the validation set) on the **held-out test set: 1,081 images from 57 videos** that played no part in training or model selection. Reports precision, recall, mAP50, mAP50-95 overall and per class, and the speed per image (one image at a time, T4 GPU, after warm-up). Saved to `OceanWatch/runs/final_test_results.csv`.

In [ ]:
# ===== 5. FINAL TEST: both models, same test set, same GPU =====
import numpy as np, pandas as pd
from ultralytics import YOLO, RTDETR

assert yolo_best, 'YOLO best.pt not found in My Drive - upload it first, then Run all again.'
MODELS = {'YOLO26-l':  YOLO(str(yolo_best[0])),
          'RT-DETR-l': RTDETR(str(RUNS / NAME / 'weights' / 'best.pt'))}
assert MODELS['YOLO26-l'].names == dict(enumerate(NAMES)), \
    f"Wrong YOLO model {MODELS['YOLO26-l'].names} - use the best.pt from the yolo26l-3cls folder"
print('Using YOLO model:', yolo_best[0])
pd.set_option('display.float_format', '{:.3f}'.format)
RES4 = RUNS / 'final_test_results.csv'

if RES4.exists():
    print('Final test already done - saved results:\n')
    print(pd.read_csv(RES4, index_col=0))
else:
    assert torch.cuda.is_available(), 'The final test needs a GPU (for fair speed numbers): Runtime -> Change runtime type -> T4 GPU.'
    test_imgs = sorted((OUT / 'images' / 'test').glob('*.jpg'))[:200]
    rows = []
    for name, m in MODELS.items():
        # Accuracy on the held-out test set
        r = m.val(data=DATA, split='test', imgsz=640, batch=16, project=str(RUNS),
                  name=f'test-{name}', exist_ok=True, verbose=False)
        row = {'model': name, 'params (M)': sum(p.numel() for p in m.model.parameters()) / 1e6,
               'precision': r.box.mp, 'recall': r.box.mr, 'mAP50': r.box.map50, 'mAP50-95': r.box.map}
        for k, c in enumerate(NAMES):
            p, rc, ap50, ap = r.box.class_result(list(r.ap_class_index).index(k))
            row[f'{c} AP50'], row[f'{c} AP50-95'] = ap50, ap
        # Speed: one image at a time (like a live camera), after warm-up
        for _ in range(10):
            m.predict(test_imgs[0], imgsz=640, verbose=False)
        ms = [sum(m.predict(p, imgsz=640, verbose=False)[0].speed.values()) for p in test_imgs]
        row['latency (ms/img)'], row['FPS'] = np.mean(ms), 1000 / np.mean(ms)
        rows.append(row)
    df = pd.DataFrame(rows).set_index('model').T
    print(df)
    df.to_csv(RES4)
    print('\nSaved to Drive:', RES4)

## 6. Experiments: image conditions and enhancement
- **Clear vs challenging:** test images are split at the median RMS contrast of the grey image; below the median = murky / challenging.
- **Enhancement:** each test image is also tested after **CLAHE** (contrast-limited adaptive histogram equalisation on the lightness channel) and after **grey-world white balance + CLAHE**. The models are not retrained.

Saved to `OceanWatch/runs/experiment_conditions_enhancement.csv`.

In [ ]:
# ===== 6. EXPERIMENTS: clear vs challenging images, and image enhancement (~15 min) =====
import cv2, yaml

EXP = WORK / 'experiments'
EXP.mkdir(exist_ok=True)
RES5 = RUNS / 'experiment_conditions_enhancement.csv'

def gray_world(img):                       # simple white balance: removes the blue/green colour cast
    f = img.astype(np.float32)
    means = f.reshape(-1, 3).mean(0)
    return np.clip(f * (means.mean() / np.maximum(means, 1e-6)), 0, 255).astype(np.uint8)

def clahe(img):                            # contrast enhancement on the lightness channel only
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

ENHANCE = {'original': None, 'CLAHE': clahe, 'white balance + CLAHE': lambda im: clahe(gray_world(im))}
test_imgs_all = sorted((OUT / 'images' / 'test').glob('*.jpg'))

# 1. Image condition: RMS contrast of the grey image. Below the median = murky / challenging.
contrast = {p.name: cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2GRAY).std() / 255 for p in test_imgs_all}
thr = float(np.median(list(contrast.values())))
GROUPS = {'all': sorted(contrast),
          'clear': sorted(n for n, c in contrast.items() if c >= thr),
          'challenging': sorted(n for n, c in contrast.items() if c < thr)}
print(f"Median RMS contrast = {thr:.3f} -> {len(GROUPS['clear'])} clear images, "
      f"{len(GROUPS['challenging'])} challenging images")

# 2. Enhanced copies of the test images (labels unchanged)
roots = {}
for i, (ename, fn) in enumerate(ENHANCE.items()):
    root = OUT if fn is None else EXP / f'enh{i}'
    if fn is not None and not (root / 'images' / 'test').exists():
        (root / 'images' / 'test').mkdir(parents=True)
        (root / 'labels' / 'test').mkdir(parents=True)
        for p in test_imgs_all:
            cv2.imwrite(str(root / 'images' / 'test' / p.name), fn(cv2.imread(str(p))),
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            shutil.copy2(OUT / 'labels' / 'test' / f'{p.stem}.txt', root / 'labels' / 'test' / f'{p.stem}.txt')
    roots[ename] = root

def subset_yaml(root, names, tag):
    lst = EXP / f'{tag}.txt'
    lst.write_text('\n'.join(str(root / 'images' / 'test' / n) for n in names))
    y = EXP / f'{tag}.yaml'
    y.write_text(yaml.safe_dump({'path': str(EXP), 'train': str(lst), 'val': str(lst), 'test': str(lst),
                                 'names': dict(enumerate(NAMES))}, sort_keys=False))
    return str(y)

def trash_ap50(r):
    idx = list(r.ap_class_index)
    return r.box.class_result(idx.index(0))[2] if 0 in idx else float('nan')

# 3. Evaluate both models on every (image group x enhancement) combination
if not RES5.exists():
    assert torch.cuda.is_available(), 'The experiments need a GPU: Runtime -> Change runtime type -> T4 GPU.'
if RES5.exists():
    res = pd.read_csv(RES5)
    print('Experiments already done - loaded saved results.')
else:
    rows = []
    for i, (ename, root) in enumerate(roots.items()):
        for j, (gname, names) in enumerate(GROUPS.items()):
            y = subset_yaml(root, names, f'e{i}_g{j}')
            for mname, m in MODELS.items():
                r = m.val(data=y, split='test', imgsz=640, batch=16, plots=False, verbose=False)
                rows.append({'model': mname, 'images': gname, 'enhancement': ename, 'n_images': len(names),
                             'mAP50': r.box.map50, 'mAP50-95': r.box.map, 'trash AP50': trash_ap50(r)})
    res = pd.DataFrame(rows)
    res.to_csv(RES5, index=False)

order = {'images': ['all', 'clear', 'challenging'], 'enhancement': list(ENHANCE)}
for metric in ['trash AP50', 'mAP50']:
    t = res.pivot_table(index=['images', 'enhancement'], columns='model', values=metric)
    t = t.reindex(pd.MultiIndex.from_product([order['images'], order['enhancement']]))
    print(f'\n=== {metric} ===')
    print(t[['YOLO26-l', 'RT-DETR-l']])
print('\nSaved to Drive:', RES5)

## 7. Figures
Finds test images where one model detects more of the real litter (IoU ≥ 0.5) than the other, and saves side-by-side panels (ground truth | YOLO | RT-DETR) to `OceanWatch/figures`.

In [ ]:
# ===== 7. FIGURES: test images where the two models differ most =====
import matplotlib.pyplot as plt

FIG = DRIVE / 'OceanWatch' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)

def load_gt(p):
    h, w = cv2.imread(str(p)).shape[:2]
    gt = []
    for line in (OUT / 'labels' / 'test' / f'{p.stem}.txt').read_text().splitlines():
        c, cx, cy, bw, bh = map(float, line.split())
        gt.append((int(c), (cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h))
    return gt

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def trash_found(res, gt):                  # ground-truth trash boxes matched at IoU >= 0.5
    pred = [b for c, b in zip(res.boxes.cls.tolist(), res.boxes.xyxy.tolist()) if int(c) == 0]
    return sum(any(iou(b, g[1:]) >= 0.5 for b in pred) for g in gt if g[0] == 0)

def draw_gt(p, gt):
    img = cv2.imread(str(p))
    for c, *xyxy in gt:
        x0, y0, x1, y1 = map(int, xyxy)
        cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)
        cv2.putText(img, NAMES[c], (x0, max(12, y0 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return img[..., ::-1]

scores = []
for p in test_imgs_all:
    gt = load_gt(p)
    if not any(g[0] == 0 for g in gt):
        continue
    ry, rr = (MODELS[k].predict(str(p), conf=0.25, verbose=False)[0] for k in ('YOLO26-l', 'RT-DETR-l'))
    scores.append((trash_found(rr, gt) - trash_found(ry, gt), p.name))
scores.sort()
picks = {'rtdetr_better': [n for d, n in scores[::-1] if d > 0][:3],
         'yolo_better':   [n for d, n in scores if d < 0][:3]}
print(f'{sum(d > 0 for d, _ in scores)} trash images where RT-DETR finds more litter, '
      f'{sum(d < 0 for d, _ in scores)} where YOLO finds more, out of {len(scores)}')

for tag, names in picks.items():
    if not names:
        continue
    fig, axes = plt.subplots(len(names), 3, figsize=(15, 3.2 * len(names)), squeeze=False)
    for row, n in zip(axes, names):
        p = OUT / 'images' / 'test' / n
        panels = [('Ground truth', draw_gt(p, load_gt(p)))]
        panels += [(k, MODELS[k].predict(str(p), conf=0.25, verbose=False)[0].plot()[..., ::-1])
                   for k in ('YOLO26-l', 'RT-DETR-l')]
        for ax, (title, im) in zip(row, panels):
            ax.imshow(im); ax.set_title(f'{title}   ({n})' if title == 'Ground truth' else title, fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(FIG / f'examples_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()
print('Saved figures to Drive:', FIG)

## 8. OceanWatch app
A web app comparing both models. Upload an underwater image (or click an example) and press **Detect litter** to see:
- both models' boxes, classes and **confidence scores**, the litter count and the time taken;
- for TrashCan test images, the **ground truth** and per-image **precision / recall** (correct, false alarms, missed);
- the overall test-set accuracy of both models.

It prints a public `https://….gradio.live` link (valid while this notebook runs, up to 72 hours). Works on CPU.

In [ ]:
# ===== 8. OCEANWATCH APP: upload an underwater image -> detections, confidence, accuracy =====
import time, numpy as np, pandas as pd, cv2
from pathlib import Path
try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gradio'], check=True)
    import gradio as gr
from ultralytics import YOLO, RTDETR

if 'MODELS' not in globals():          # Cell 4 not run in this session: load the two trained models
    MODELS = {'YOLO26-l':  YOLO(str(yolo_best[0])),
              'RT-DETR-l': RTDETR(str(RUNS / NAME / 'weights' / 'best.pt'))}
COLORS = {0: (230, 126, 34), 1: (46, 204, 113), 2: (52, 152, 219)}   # trash orange, bio green, rov blue (RGB)

def app_iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def find_ground_truth(path, h, w):
    """If the image is from TrashCan, return (split, boxes) from its label file, else (None, [])."""
    stem = Path(path).stem
    for split in ('test', 'val', 'train'):
        f = OUT / 'labels' / split / f'{stem}.txt'
        if f.exists():
            boxes = []
            for line in f.read_text().splitlines():
                c, cx, cy, bw, bh = map(float, line.split())
                boxes.append((int(c), [(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h]))
            return split, boxes
    return None, []

def match(preds, gt, thr=0.5):
    """Greedy matching by confidence: a prediction is correct if same class and IoU >= 0.5."""
    used, tp = set(), 0
    for c, box, conf in sorted(preds, key=lambda p: -p[2]):
        best, best_iou = None, thr
        for j, (gc, gbox) in enumerate(gt):
            if j not in used and gc == c and app_iou(box, gbox) >= best_iou:
                best, best_iou = j, app_iou(box, gbox)
        if best is not None:
            used.add(best); tp += 1
    return tp, len(preds) - tp, len(gt) - tp

def draw(rgb, boxes, scores=None):
    img = rgb.copy()
    for k, (c, (x0, y0, x1, y1)) in enumerate(boxes):
        col = COLORS[c]
        cv2.rectangle(img, (int(x0), int(y0)), (int(x1), int(y1)), col, 2)
        label = NAMES[c] + (f' {scores[k]:.2f}' if scores is not None else '')
        cv2.putText(img, label, (int(x0) + 2, max(14, int(y0) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2)
    return img

def analyse(path, conf):
    if path is None:
        return None, None, None, 'Upload an underwater image or pick an example below.', None, None
    rgb = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    split, gt = find_ground_truth(path, h, w)
    images, rows, dets = [], [], []
    for name, m in MODELS.items():
        r = m.predict(path, imgsz=640, conf=conf, verbose=False)[0]
        preds = [(int(c), b, float(s)) for c, b, s in
                 zip(r.boxes.cls.tolist(), r.boxes.xyxy.tolist(), r.boxes.conf.tolist())]
        images.append(draw(rgb, [(c, b) for c, b, _ in preds], [s for *_, s in preds]))
        row = {'Model': name,
               'Litter found': sum(c == 0 for c, *_ in preds),
               'Marine life': sum(c == 1 for c, *_ in preds),
               'ROV parts': sum(c == 2 for c, *_ in preds),
               'Avg confidence': f"{np.mean([s for *_, s in preds]):.2f}" if preds else '-',
               'Time (ms)': f"{sum(r.speed.values()):.0f}"}
        if gt:
            tp, fp, fn = match(preds, gt)
            row.update({'Correct (TP)': tp, 'False alarms (FP)': fp, 'Missed (FN)': fn,
                        'Precision': f'{tp / (tp + fp):.0%}' if tp + fp else '-',
                        'Recall': f'{tp / (tp + fn):.0%}' if tp + fn else '-'})
        rows.append(row)
        dets += [{'Model': name, 'Class': NAMES[c], 'Confidence': f'{s:.2f}',
                  'Box (x1, y1, x2, y2)': ', '.join(f'{v:.0f}' for v in b)} for c, b, s in preds]
    gt_img = draw(rgb, gt) if gt else None
    if split == 'test':
        note = f'**Ground truth found** (test image, never seen in training): {len(gt)} labelled objects. Accuracy below is measured against it.'
    elif split:
        note = f'**Warning:** this image is from the **{split}** set, so the models saw it (or its video) during training. Its accuracy is not a fair test.'
    else:
        note = 'No ground-truth labels for this image (not from the dataset), so per-image accuracy cannot be measured. The overall test-set accuracy is shown below.'
    return images[0], images[1], gt_img, note, pd.DataFrame(rows), pd.DataFrame(dets)

overall = pd.read_csv(RUNS / 'final_test_results.csv', index_col=0) if (RUNS / 'final_test_results.csv').exists() else None
if overall is not None:
    keep = ['precision', 'recall', 'mAP50', 'mAP50-95', 'trash AP50', 'bio AP50', 'rov AP50', 'latency (ms/img)', 'FPS']
    overall = overall.loc[[k for k in keep if k in overall.index]].astype(float).round(3).reset_index().rename(columns={'index': 'Test-set metric'})

test_dir = OUT / 'images' / 'test'
trash_imgs = [p for p in sorted(test_dir.glob('*.jpg'))
              if any(l.startswith('0 ') for l in (OUT / 'labels' / 'test' / f'{p.stem}.txt').read_text().splitlines())]
examples = [[str(p), 0.25] for p in trash_imgs[::max(1, len(trash_imgs) // 6)][:6]]

with gr.Blocks(title='OceanWatch') as demo:
    gr.Markdown('# 🌊 OceanWatch: underwater marine litter detection\n'
                'Compares **YOLO26-l** (CNN) and **RT-DETR-l** (transformer), both trained on TrashCan 1.0. '
                'Classes: <span style="color:#e67e22">**trash**</span>, <span style="color:#2ecc71">**bio**</span> (marine life), '
                '<span style="color:#3498db">**rov**</span> (robot parts).')
    with gr.Row():
        inp = gr.Image(type='filepath', label='Underwater image', height=320)
        with gr.Column():
            conf = gr.Slider(0.05, 0.9, value=0.25, step=0.05, label='Confidence threshold')
            btn = gr.Button('Detect litter', variant='primary')
            note = gr.Markdown()
    with gr.Row():
        out_y = gr.Image(label='YOLO26-l prediction')
        out_r = gr.Image(label='RT-DETR-l prediction')
        out_gt = gr.Image(label='Ground truth (if available)')
    gr.Markdown('### Results for this image')
    table = gr.Dataframe(wrap=True)
    gr.Markdown('### Every detection with its confidence score')
    det_table = gr.Dataframe(wrap=True)
    if overall is not None:
        gr.Markdown('### Overall accuracy on the full test set (1,081 unseen images)')
        gr.Dataframe(overall, wrap=True)
    gr.Examples(examples, [inp, conf], label='Test images (with ground truth): click one, then Detect litter')
    btn.click(analyse, [inp, conf], [out_y, out_r, out_gt, note, table, det_table])

demo.launch(share=True, debug=False)   # prints a public https://....gradio.live link; keep this notebook running

## 9. Export files for the Streamlit app (GitHub)
Collects everything the Streamlit app needs into **Google Drive → OceanWatch → streamlit_files**: the two trained models (renamed), 12 test images with their ground-truth labels, and the results tables. Runs on CPU in under a minute. Needs Steps 1, 2 and 4 to have run in this session (Step 4 just says "already finished").

In [ ]:
# ===== 9. EXPORT files for the Streamlit app =====
import random, shutil

EXPORT = DRIVE / 'OceanWatch' / 'streamlit_files'
for sub in ('models', 'samples', 'results'):
    (EXPORT / sub).mkdir(parents=True, exist_ok=True)

# 1. Trained models (uploaded to the GitHub Release, not the repository itself)
shutil.copy2(yolo_best[0], EXPORT / 'models' / 'yolo26l_3cls.pt')
shutil.copy2(RUNS / 'rtdetr-l-3cls' / 'weights' / 'best.pt', EXPORT / 'models' / 'rtdetr_l_3cls.pt')

# 2. Results tables
for f in ('final_test_results.csv', 'experiment_conditions_enhancement.csv'):
    if (RUNS / f).exists():
        shutil.copy2(RUNS / f, EXPORT / 'results' / f)

# 3. Twelve test images that contain litter, with their ground-truth labels
test_lbl = OUT / 'labels' / 'test'
with_trash = [p for p in sorted((OUT / 'images' / 'test').glob('*.jpg'))
              if any(l.startswith('0 ') for l in (test_lbl / f'{p.stem}.txt').read_text().splitlines())]
for p in random.Random(0).sample(with_trash, 12):
    shutil.copy2(p, EXPORT / 'samples' / p.name)
    shutil.copy2(test_lbl / f'{p.stem}.txt', EXPORT / 'samples' / f'{p.stem}.txt')

for sub in ('models', 'samples', 'results'):
    files = sorted((EXPORT / sub).iterdir())
    print(f'{sub:<8} {len(files):>2} files, {sum(f.stat().st_size for f in files) / 1e6:.1f} MB')
print('\nSaved to Google Drive:', EXPORT)